# Combine and clean human RPS data

Combines two human-vs-human rock-paper-scissors datasets into one tidy table for transition-matrix training.

| Source | Raw file | Players | Rounds |
|---|---|---|---|
| Brockbank & Vul (2021), *Games* 12(3):70 — [repo](https://github.com/erik-brockbank/rps-games-analysis) | `data/raw/rps_v1_data.csv` | 124 (62 fixed pairs) | 300 per game |
| Wang, Xu & Zhou (2014), *Sci. Rep.* 4:5830 — via R package `stratEst` (GPL-3) | `data/raw/WXZ2014.rda` | 72 (random re-matching in groups of 6) | 300 |

**Output:** `data/processed/rps_human_combined.csv`, one row per player per round:

| column | meaning |
|---|---|
| `dataset` | `brockbank` or `wxz` |
| `game_id`, `player_id` | unique across datasets |
| `round` | 1–300 |
| `move`, `opp_move` | `R` / `P` / `S` |
| `outcome` | `win` / `lose` / `tie` for this player |
| `prev_move`, `prev_opp_move`, `prev_outcome` | previous round for the same player; empty on round 1 or after an invalid round |

In [1]:
from pathlib import Path

import pandas as pd
import pyreadr

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent  # repo root or notebooks/
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

MOVES = ["R", "P", "S"]
BEATS = {"R": "S", "P": "R", "S": "P"}  # key beats value


def outcome(move, opp):
    if move == opp:
        return "tie"
    return "win" if BEATS[move] == opp else "lose"

## 1. Brockbank & Vul (2021)

One row per player per round. Two players per `game_id`.

In [2]:
bb_raw = pd.read_csv(RAW / "rps_v1_data.csv")
print(bb_raw.shape)
print(bb_raw["player_move"].value_counts(dropna=False))
bb_raw.groupby("game_id")["round_index"].max().value_counts()

(36030, 10)
player_move
scissors    12444
paper       12068
rock        11457
none           31
NaN            30
Name: count, dtype: int64


round_index
300    58
52      1
53      1
220     1
290     1
Name: count, dtype: int64

Issues found:
- 4 games stop early (52, 53, 220 and 290 rounds). Drop them, because they are probably disconnects.
- `player_move` is `none` (timeout, scored as a loss) or NaN (round not recorded) in a few rows. Treat both as invalid.

Each player's row is joined with the opponent's row from the same round. That gives `opp_move` and lets us recompute `outcome`.

In [3]:
bb = bb_raw.copy()
rounds_per_game = bb.groupby("game_id")["round_index"].transform("max")
bb = bb[rounds_per_game == 300]

bb["move"] = bb["player_move"].map({"rock": "R", "paper": "P", "scissors": "S"})  # none/NaN -> NaN

opp = bb[["game_id", "round_index", "player_id", "move"]].rename(
    columns={"player_id": "opp_id", "move": "opp_move"})
bb = bb.merge(opp, on=["game_id", "round_index"])
bb = bb[bb["player_id"] != bb["opp_id"]]

bb = bb.rename(columns={"round_index": "round"})
bb["dataset"] = "brockbank"
bb = bb[["dataset", "game_id", "player_id", "round", "move", "opp_move"]]
print(bb.shape, "games:", bb["game_id"].nunique())

(34800, 6) games: 58


## 2. Wang, Xu & Zhou (2014)

One row per player per period. The opponent is re-drawn every round, so each player is treated as their own `game_id`.

In [4]:
wxz_raw = pyreadr.read_r(RAW / "WXZ2014.rda")["WXZ2014"]
print(wxz_raw.shape)
print(wxz_raw.isna().sum().sum(), "missing values")
wxz_raw.head()

(21600, 6)
0 missing values


,id,game,period,choice,other.choice,result
0,1.0,1.0,1,s,s,tie
1,1.0,1.0,2,s,r,lose
2,1.0,1.0,3,s,p,win
3,1.0,1.0,4,s,p,win
4,1.0,1.0,5,s,p,win


In [5]:
wxz = pd.DataFrame({
    "dataset": "wxz",
    "game_id": "wxz_" + wxz_raw["id"].astype(int).astype(str),
    "player_id": "wxz_" + wxz_raw["id"].astype(int).astype(str),
    "round": wxz_raw["period"].astype(int),
    "move": wxz_raw["choice"].astype(str).str.upper(),
    "opp_move": wxz_raw["other.choice"].astype(str).str.upper(),
})

# the provided result column should agree with the outcome recomputed from the moves
recomputed = [outcome(m, o) for m, o in zip(wxz["move"], wxz["opp_move"])]
assert (pd.Series(recomputed) == wxz_raw["result"].astype(str).values).all()
print(wxz.shape, "players:", wxz["player_id"].nunique())

(21600, 6) players: 72


## 3. Combine, add previous-round features, drop invalid rounds

In [6]:
df = pd.concat([bb, wxz], ignore_index=True)
assert not df.duplicated(["player_id", "round"]).any()

df["valid"] = df["move"].isin(MOVES) & df["opp_move"].isin(MOVES)
df["outcome"] = None
df.loc[df["valid"], "outcome"] = [outcome(m, o) for m, o in df.loc[df["valid"], ["move", "opp_move"]].values]

# lag on the full sequence first, so a round after an invalid one gets an empty history
df = df.sort_values(["dataset", "player_id", "round"]).reset_index(drop=True)
prev_valid = df.groupby("player_id")["valid"].shift(1).eq(True)
for col in ["move", "opp_move", "outcome"]:
    df["prev_" + col] = df.groupby("player_id")[col].shift(1).where(prev_valid)

print("invalid rounds dropped:", (~df["valid"]).sum())
df = df[df["valid"]].drop(columns="valid").reset_index(drop=True)
df.head()

invalid rounds dropped: 86


,dataset,game_id,player_id,round,move,opp_move,outcome,prev_move,prev_opp_move,prev_outcome
0,brockbank,23460664-eee3-45dd-997c-49595d3aeb2e,0333c73a-6d1f-4cc9-afde-d0d833bc47f8,1,R,P,lose,NaN,NaN,NaN
1,brockbank,23460664-eee3-45dd-997c-49595d3aeb2e,0333c73a-6d1f-4cc9-afde-d0d833bc47f8,2,R,S,win,R,P,lose
2,brockbank,23460664-eee3-45dd-997c-49595d3aeb2e,0333c73a-6d1f-4cc9-afde-d0d833bc47f8,3,P,R,win,R,S,win
3,brockbank,23460664-eee3-45dd-997c-49595d3aeb2e,0333c73a-6d1f-4cc9-afde-d0d833bc47f8,5,P,P,tie,NaN,NaN,NaN
4,brockbank,23460664-eee3-45dd-997c-49595d3aeb2e,0333c73a-6d1f-4cc9-afde-d0d833bc47f8,6,R,S,win,P,P,tie


## 4. Sanity checks

In [7]:
print(df.groupby("dataset").agg(players=("player_id", "nunique"), rows=("move", "size")))
print()
print(df["move"].value_counts(normalize=True).round(3))
print()
print(df["outcome"].value_counts(normalize=True).round(3))
assert (df["prev_move"].isna() == df["prev_outcome"].isna()).all()

           players   rows
dataset                  
brockbank      116  34714
wxz             72  21600

move
S    0.336
R    0.333
P    0.330
Name: proportion, dtype: float64

outcome
lose    0.335
win     0.335
tie     0.329
Name: proportion, dtype: float64


Quick look at the target: P(next move | previous outcome, previous move). Rows with no history are skipped.

In [8]:
hist = df.dropna(subset=["prev_move", "prev_outcome"])
pd.crosstab([hist["prev_outcome"], hist["prev_move"]], hist["move"], normalize="index").round(3)

move                        P      R      S
prev_outcome prev_move                     
lose         P          0.318  0.322  0.360
             R          0.338  0.356  0.306
             S          0.351  0.285  0.363
tie          P          0.319  0.358  0.323
             R          0.292  0.373  0.334
             S          0.351  0.298  0.351
win          P          0.327  0.299  0.374
             R          0.354  0.352  0.294
             S          0.319  0.354  0.326

## 5. Save

In [9]:
path = OUT / "rps_human_combined.csv"
df.to_csv(path, index=False)
print(path, df.shape)

D:\SMU\CS103\proj\rps-transition-matrix\data\processed\rps_human_combined.csv (56314, 10)


## 6. Partitions

`rps_human_combined.csv` stays the master. Train/test copies go to `data/processed/partitions/`: per dataset, 20% of players (rounded down, seed 103) go to test, the rest to train. Pooled = brockbank + wxz of the same split.

In [10]:
import sys

sys.path.insert(0, str(ROOT / "src"))
from rps_markov.partition import write_partitions

write_partitions(OUT / "rps_human_combined.csv", OUT / "partitions")
for f in sorted((OUT / "partitions").glob("*.csv")):
    print(f.name, pd.read_csv(f).shape)

test_brockbank.csv (6877, 10)
test_pooled.csv (11077, 10)
test_wxz.csv (4200, 10)
train_brockbank.csv (27837, 10)


train_pooled.csv (45237, 10)
train_wxz.csv (17400, 10)
